<a href="https://colab.research.google.com/github/Martin2513/fives-segmentation-unet-poc/blob/main/unet_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import zipfile
from google.colab import drive
from tqdm.auto import tqdm


In [2]:
drive.mount("/content/drive")

ZIP_PATH = "/content/drive/MyDrive/FIVES.zip"
DATA_DIR = "data"

if not os.path.exists(DATA_DIR):
  with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for file in tqdm(z.infolist(), desc="Extracting FIVES"):
      z.extract(file, DATA_DIR)

  print("Extracted to directory:", DATA_DIR)

Mounted at /content/drive


Extracting FIVES:   0%|          | 0/1608 [00:00<?, ?it/s]

Extracted to directory: data


In [3]:
import cv2

In [4]:
IMG_SIZE = 768
RESIZED_DIR = "data_768"

# Find the root folder in "data"
#(where train/Original and test/Original are)
DATA_ROOT = None
for root, dirs, files in os.walk(DATA_DIR):
    if os.path.isdir(os.path.join(root, "train", "Original")) and os.path.isdir(os.path.join(root, "test", "Original")):
        DATA_ROOT = root
        break

assert DATA_ROOT is not None, "couldn't find the folders train/Original and test/Original"
print("Dataset found in:", DATA_ROOT)

# Resize and save one split ("train" or "test")
def resize_and_cache(split_name):
    src_img_dir = os.path.join(DATA_ROOT, split_name, "Original")
    src_msk_dir = os.path.join(DATA_ROOT, split_name, "Ground truth")
    dst_img_dir = os.path.join(RESIZED_DIR, split_name, "Original")
    dst_msk_dir = os.path.join(RESIZED_DIR, split_name, "Ground truth")

    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_msk_dir, exist_ok=True)

    files = sorted([f for f in os.listdir(src_img_dir) if f.lower().endswith(".png")])
    for fname in tqdm(files, desc=f"Processing {split_name}"):
        dst_img = os.path.join(dst_img_dir, fname)
        dst_msk = os.path.join(dst_msk_dir, fname)
        if os.path.exists(dst_img) and os.path.exists(dst_msk):
            continue

        img = cv2.imread(os.path.join(src_img_dir, fname), cv2.IMREAD_COLOR)
        msk = cv2.imread(os.path.join(src_msk_dir, fname), cv2.IMREAD_GRAYSCALE)

        img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        msk_resized = cv2.resize(msk, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        msk_resized = ((msk_resized > 127).astype("uint8")) * 255

        cv2.imwrite(dst_img, img_resized)
        cv2.imwrite(dst_msk, msk_resized)

    return len(files)


n_train = resize_and_cache("train")
n_test = resize_and_cache("test")
print(f"Saved {n_train} train and {n_test} test images in '{RESIZED_DIR}'.")

Dataset found in: data


Processing train:   0%|          | 0/600 [00:00<?, ?it/s]

Processing test:   0%|          | 0/200 [00:00<?, ?it/s]

Saved 600 train and 200 test images in 'data_768'.
